# Mini traductor: entender el Transformer
Ejecuta las celdas en orden. Necesitas PyTorch instalado en el entorno de Jupyter; para usar la RTX 4050, debe ser una versión con CUDA ([instalación oficial](https://pytorch.org/get-started/locally/)).

El objetivo es seguir la arquitectura de un Transformer pequeño mientras aprende 120 frases. Las traducciones finales usan ejemplos del entrenamiento: muestran lo aprendido, pero no evalúan generalización.

```text
Español → Embedding + posición → Encoder → memoria
                                             ↓
Inglés  → Embedding + posición → Decoder → Linear → siguiente palabra
                                      ↑
                                máscara causal
```

## 1. Importaciones y dispositivo
El modelo y los datos usarán la GPU si CUDA está disponible.

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


device = "cuda"

## 2. Frases de ejemplo
Escribimos las conjugaciones directamente. Después añadimos seis objetos a cada inicio de frase: **20 × 6 = 120 pares**.

Todas las frases tienen cuatro palabras; así podemos agruparlas sin añadir padding.

In [3]:
inicios = [
    ("yo veo", "i see"),
    ("tú ves", "you see"),
    ("él ve", "he sees"),
    ("ella ve", "she sees"),
    ("nosotros vemos", "we see"),
    ("yo amo", "i love"),
    ("tú amas", "you love"),
    ("él ama", "he loves"),
    ("ella ama", "she loves"),
    ("nosotros amamos", "we love"),
    ("yo tengo", "i have"),
    ("tú tienes", "you have"),
    ("él tiene", "he has"),
    ("ella tiene", "she has"),
    ("nosotros tenemos", "we have"),
    ("yo compro", "i buy"),
    ("tú compras", "you buy"),
    ("él compra", "he buys"),
    ("ella compra", "she buys"),
    ("nosotros compramos", "we buy"),
]

objetos = [
    ("el gato", "the cat"),
    ("el perro", "the dog"),
    ("el libro", "the book"),
    ("la casa", "the house"),
    ("la pelota", "the ball"),
    ("la bicicleta", "the bicycle"),
]

pares = []
for inicio_es, inicio_en in inicios:
    for objeto_es, objeto_en in objetos:
        pares.append((inicio_es + " " + objeto_es, inicio_en + " " + objeto_en))

print("Número de pares:", len(pares))
print(pares[:3])

Número de pares: 120
[('yo veo el gato', 'i see the cat'), ('yo veo el perro', 'i see the dog'), ('yo veo el libro', 'i see the book')]


## 3. Palabras → números
`split()` separa las palabras por espacios. Cada idioma tiene su propio vocabulario.

En inglés añadimos `<bos>` (inicio) y `<eos>` (fin).

In [5]:
palabras_es = sorted({palabra for es, en in pares for palabra in es.split()})
palabras_en = ["<ini>", "<fin>"] + sorted(
    {palabra for es, en in pares for palabra in en.split()}
)

vocab_es = {palabra: i for i, palabra in enumerate(palabras_es)}
vocab_en = {palabra: i for i, palabra in enumerate(palabras_en)}

INICIO = vocab_en["<ini>"]
FINAL = vocab_en["<fin>"]

def codificar(frase, vocabulario):
    return [vocabulario[palabra] for palabra in frase.lower().split()]

print("yo veo el gato →", codificar("yo veo el gato", vocab_es))

yo veo el gato → [27, 25, 10, 12]


## 4. Datos para el entrenamiento
El encoder recibe español. El decoder recibe inglés con un token de inicio y aprende a predecir la palabra siguiente.

```text
Entrada del encoder:  yo veo el gato
Entrada del decoder:  <bos> i   see the cat
Objetivo:              i    see the cat <eos>
```

Durante el entrenamiento usamos las palabras correctas como entrada del decoder (**teacher forcing**). El desplazamiento se hace con `[:-1]` y `[1:]`.

In [ ]:
src = torch.tensor([codificar(es, vocab_es) for es, en in pares])  # 120 frases  4 palabras por frase [120, 4]
tgt = torch.tensor([[INICIO] + codificar(en, vocab_en) + [FINAL] for es, en in pares]) # 120 frases  4 palabras por frase [120, 6]

loader = DataLoader(TensorDataset(src, tgt), batch_size=32, shuffle=True)

print("Encoder:", src[0].tolist())
print("Decoder:", [palabras_en[i] for i in tgt[0, :-1].tolist()])
print("Objetivo:", [palabras_en[i] for i in tgt[0, 1:].tolist()])

Encoder: [27, 25, 10, 12]
Decoder: ['<ini>', 'i', 'see', 'the', 'cat']
Objetivo: ['i', 'see', 'the', 'cat', '<fin>']


## 5. Máscara causal
El decoder puede consultar las palabras anteriores y la posición actual. Las posiciones futuras se bloquean.

En esta máscara booleana, **1 = bloqueado** y **0 = permitido**. Las filas consultan a las columnas.

In [7]:
def mascara_causal(longitud):
    return torch.ones(longitud, longitud, dtype=torch.bool, device=device).triu(1)

print(mascara_causal(5).int())

tensor([[0, 1, 1, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0]], device='cuda:0', dtype=torch.int32)


## 6. Arquitectura del Transformer
Esta es la parte central del notebook.

- **Embedding:** convierte cada número en un vector de 64 valores.
- **Posición:** sumamos un vector aprendido para indicar el lugar de cada palabra.
- **Encoder:** procesa la frase española y produce la memoria.
- **Decoder:** procesa el inglés, aplica la máscara causal y consulta la memoria.
- **Linear:** convierte los vectores del decoder en puntuaciones para el vocabulario inglés.

Usamos un bloque de encoder y uno de decoder, con cuatro cabezas de atención. Dentro de esos bloques, `nn.Transformer` implementa:

```text
Encoder:
  Self-attention → Add & Norm → Feed-forward → Add & Norm

Decoder:
  Self-attention causal → Add & Norm
  Atención a la memoria del encoder → Add & Norm
  Feed-forward → Add & Norm
```

`batch_first=True` significa **[lote, número de palabras, dimensión]**. Las posiciones son embeddings aprendidos; el artículo original usa posiciones sinusoidales. [Documentación de nn.Transformer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html).

In [ ]:
class MiniTraductor(nn.Module):
    def __init__(self):
        super().__init__()
        # 29 * 64 
        self.embedding_es = nn.Embedding(len(vocab_es), 64) # "yo" = 27   ->  27=[1.,5,2,5,....64]
        self.embedding_en = nn.Embedding(len(vocab_en), 64)
        self.posicion = nn.Embedding(10, 64)

        self.transformer = nn.Transformer(
            d_model=64,
            nhead=4,
            encoder_layers=1,
            decoder_layers=1,
            dim_feedforward=128,
        )

        self.linear = nn.Linear(64, len(vocab_en)) #64 -> 22

    def forward(self, src, tgt):
        # 1. Palabras + posiciones
        posiciones_src = torch.arange(src.size(1), device=src.device)
        posiciones_tgt = torch.arange(tgt.size(1), device=tgt.device)

        src = self.embedding_es(src) + self.posicion(posiciones_src)
        tgt = self.embedding_en(tgt) + self.posicion(posiciones_tgt)

        # 2. El encoder construye la memoria del español
        memoria = self.transformer.encoder(src)

        # 3. El decoder consulta esa memoria sin mirar palabras futuras
        salida = self.transformer.decoder(
            tgt,
            memoria,
            tgt_mask=mascara_causal(tgt.size(1)),
        )

        # 4. Puntuaciones para cada palabra del vocabulario inglés
        return self.linear(salida)

modelo = MiniTraductor().to(device)

## 7. Entrenamiento
`CrossEntropyLoss` compara las puntuaciones con la palabra correcta. Recibe las puntuaciones directamente, sin aplicar softmax antes.

`backward()` calcula los gradientes y `step()` actualiza los pesos. `reshape` agrupa las posiciones de todos los ejemplos del lote para calcular la pérdida.

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)

modelo.train()

for epoca in range(200):
    perdida_total = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        entrada_decoder = tgt[:, :-1]
        objetivo = tgt[:, 1:]

        prediccion = modelo(src, entrada_decoder)
        perdida = criterio(
            prediccion.reshape(-1, len(vocab_en)),
            objetivo.reshape(-1),
        )

        optimizador.zero_grad() #acumula
        perdida.backward() # calcula cuanto ayudo al error
        optimizador.step() # utiliza los gradientes para modificar w

        perdida_total += perdida.item()

    if (epoca + 1) % 20 == 0:
        print(f"Época {epoca + 1}: pérdida = {perdida_total / len(loader):.4f}")

Época 20: pérdida = 0.1251
Época 40: pérdida = 0.0327
Época 60: pérdida = 0.0154
Época 80: pérdida = 0.0098
Época 100: pérdida = 0.0066
Época 120: pérdida = 0.0045
Época 140: pérdida = 0.0035
Época 160: pérdida = 0.0063
Época 180: pérdida = 0.0030
Época 200: pérdida = 0.0025


## 8. Generar una traducción
Ahora el decoder empieza solo con `<bos>`. Elegimos la palabra con mayor puntuación (`argmax`), la añadimos y repetimos.

Como las traducciones tienen cuatro palabras, bastan cinco pasos: cuatro palabras y `<eos>`. Por claridad, ejecutamos el encoder y el decoder de nuevo en cada paso.

In [10]:
@torch.no_grad()
def traducir(frase):
    modelo.eval()

    src = torch.tensor([codificar(frase, vocab_es)], device=device)
    generados = [INICIO]

    for paso in range(5):
        tgt = torch.tensor([generados], device=device)
        prediccion = modelo(src, tgt)

        # Usamos la última posición para elegir la siguiente palabra
        siguiente = prediccion[0, -1].argmax().item()

        if siguiente == FINAL:
            break

        generados.append(siguiente)

    return " ".join(palabras_en[i] for i in generados[1:])

## 9. Ejemplos finales
Prueba las frases después de entrenar. Este modelo aprende el pequeño dataset; los resultados pueden variar entre ejecuciones.

In [11]:
ejemplos = [
    "yo veo el gato",
    "ella tiene el libro",
    "él ama el perro",
    "tú compras la pelota",
    "nosotros compramos la bicicleta",
]

for frase in ejemplos:
    print(frase, "→", traducir(frase))

yo veo el gato → i see the cat
ella tiene el libro → she has the book
él ama el perro → he loves the dog
tú compras la pelota → you buy the ball
nosotros compramos la bicicleta → we buy the bicycle
